# LeftHandRaise7DF Tutorial One

![Image](../../images/LeftHandRaise7DF_TutorialOne.png)

Advanced tutorial integrating control code with engineering insights.


---
<video width="640" height="480" autoplay muted loop controls>
  <source src="../../videos/g1_hand7_sdk_dds.mp4" type="video/mp4">
  Your browser does not support the video tag.
</video>

---

> ### ⚠️ **Important Notice: Environment Compatibility**

> The following Python code is sourced from the file `g1_hand7_sdk_dds_example.py`. While this script has been **successfully tested on the physical Unitree G1 humanoid hardware**, please be aware of the following:
>
> * **Hardware Only:** This specific version is designed to communicate directly with the robot's onboard computer via the **DDS (Data Distribution Service)** pipeline.
> * **Simulation Refactoring:** To run this logic within the **NVIDIA Isaac Sim** environment, significant architectural changes and code refactoring were required (including environment toggles, joint-name mapping, and simulation-app lifecycle management).
>
> For a detailed walkthrough of the refactoring process and the dual-mode (Sim-to-Real) version of this controller, please refer to the next module: 
> 📓 **`LeftHandRaise7DF_TutorialTwo.ipynb`**

---

### 🔬 Engineering Deep Dive (from Technical Report)

**Control is not just software — it is constrained by hardware physics.**

- The Unitree G1 uses a **Quasi-Direct Drive (QDD)** system with a low gear ratio (6.33:1).
- This enables **mechanical transparency** (back-drivability), allowing the robot to safely absorb impacts.
- Humanoid robots experience **2–3× body weight forces** during locomotion, requiring compliant control.

---

### ⚙️ PD Control Interpretation

The control law implemented implicitly by the robot is:

$$
\tau = K_p (q_{target} - q) + K_d (\dot{q}_{target} - \dot{q}) + \tau_{ff}
$$

This means:

- `cmd.q` defines the **equilibrium point**
- `cmd.kp` defines **how stiff the virtual spring is**
- `cmd.kd` defines **how much damping resists motion**
- `cmd.tau` adds **direct torque compensation**

---

### 🧠 Practical Engineering Implications

**cmd.q (Position)**
- Large jumps → large current spikes → potential shutdown
- Always **interpolate / ramp values**

**cmd.kp (Stiffness)**
- High → precise but dangerous (gear stress)
- Low → compliant but less accurate

**cmd.kd (Damping)**
- Prevents oscillations
- Critical for stability after motion

**cmd.tau (Feedforward)**
- Used for gravity compensation
- Reduces load on PD controller

---

### 🔥 System Constraints

**Thermal Limit**
- Heat ∝ I²R
- Holding positions generates continuous current
- Risk: thermal shutdown

**Reflected Inertia**
$$
J_{output} = J_{rotor} \times N^2
$$

- Low gear ratio → safer interaction
- High gear ratio → rigid but dangerous

---

### 🤖 RL Connection

In reinforcement learning:

```
action = policy(state)
cmd.q = action
```

The PD controller acts as a **stable interface layer** between:
- Neural policy
- Physical hardware

This is what enables **sim-to-real transfer**.



## 1. Imports and System Setup

This section initializes all required libraries and SDK components needed to communicate with the Unitree G1 robot.

To dive deeper into this section, open [this notebook](LeftHandRaise7DF_TutorialOne_Sec1.ipynb)

---

In [ ]:
import time
import sys
import numpy as np

from unitree_sdk2py.core.channel import ChannelPublisher, ChannelSubscriber, ChannelFactoryInitialize
from unitree_sdk2py.idl.unitree_hg.msg.dds_ import LowCmd_, LowState_
from unitree_sdk2py.idl.default import unitree_hg_msg_dds__LowCmd_
from unitree_sdk2py.utils.crc import CRC
from unitree_sdk2py.utils.thread import RecurrentThread

kPi = 3.141592654

# 2. Joint Index Mapping

Defines the indices of each joint in the robot's arm. These indices map software commands to physical actuators.

To dive deeper into this section, open [this notebook](LeftHandRaise7DF_TutorialOne_Sec2.ipynb)

---

class G1JointIndex:
    LeftShoulderPitch = 15
    LeftShoulderRoll = 16
    LeftShoulderYaw = 17
    LeftElbow = 18
    LeftWristRoll = 19
    LeftWristPitch = 20
    LeftWristYaw = 21

    kNotUsedJoint = 29

## 3. Controller Initialization

Initializes timing, control gains (kp, kd), and joint targets. This defines the behavior of the PD controller.

To dive deeper into this section, open [this notebook](LeftHandRaise7DF_TutorialOne_Sec3.ipynb)

---

In [ ]:
class LeftArmRaise:
    def __init__(self):
        self.dt = 0.02
        self.time_ = 0.0

        self.stage_duration = 3.0

        self.kp = 60.0
        self.kd = 1.5

        self.low_cmd = unitree_hg_msg_dds__LowCmd_()
        self.low_state = None
        self.first_update = False

        self.crc = CRC()
        self.done = False

        self.current_stage = -1

        self.joints = [
            G1JointIndex.LeftShoulderPitch,
            G1JointIndex.LeftShoulderRoll,
            G1JointIndex.LeftShoulderYaw,
            G1JointIndex.LeftElbow,
            G1JointIndex.LeftWristRoll,
            G1JointIndex.LeftWristPitch,
            G1JointIndex.LeftWristYaw,
        ]

        self.target_raise = [-0.3, 0.2, 0.0, -1.0, 0.0, 0.5, 0.0]
        self.target_extend = [-0.3, 0.2, 0.0, -1.0, 0.0, 1.0, 0.2]

        self.initial_pose = None

## 4. Communication Initialization

Sets up DDS communication channels for sending commands and receiving robot state.

To dive deeper into this section, open [this notebook](LeftHandRaise7DF_TutorialOne_Sec4.ipynb)

---

In [ ]:
    def Init(self):
        self.pub = ChannelPublisher("rt/arm_sdk", LowCmd_)
        self.pub.Init()

        self.sub = ChannelSubscriber("rt/lowstate", LowState_)
        self.sub.Init(self.LowStateHandler, 10)

## 5. State Callback

Receives robot state and captures initial joint positions.

To dive deeper into this section, open [this notebook](LeftHandRaise7DF_TutorialOne_Sec5.ipynb)

---

In [ ]:
    def LowStateHandler(self, msg):
        self.low_state = msg

        if not self.first_update:
            self.first_update = True
            self.initial_pose = [msg.motor_state[j].q for j in self.joints]

## 6. Stage Logging

Tracks and prints transitions between motion stages.

To dive deeper into this section, open [this notebook](LeftHandRaise7DF_TutorialOne_Sec6.ipynb)

---

    def log_stage(self, stage, description):
        if self.current_stage != stage:
            self.current_stage = stage
            print(f"\n[Time {self.time_:.2f}s] Stage {stage}: {description}")

## 7. Control Loop Start

Starts the periodic control loop thread once initial state is received.

To dive deeper into this section, open [this notebook](LeftHandRaise7DF_TutorialOne_Sec7.ipynb)

---

In [ ]:
    def Start(self):
        while not self.first_update:
            time.sleep(0.5)

        self.thread = RecurrentThread(
            interval=self.dt,
            target=self.ControlLoop,
            name="arm_control"
        )
        self.thread.Start()

## 8. Interpolation Function

Smoothly transitions between joint positions using linear interpolation.

To dive deeper into this section, open [this notebook](LeftHandRaise7DF_TutorialOne_Sec8.ipynb)

---

In [ ]:
    def interp(self, a, b, r):
        return (1 - r) * a + r * b

## 9. Control Loop

Core logic that updates joint commands at each timestep using staged motion.

To dive deeper into this section, open [this notebook](LeftHandRaise7DF_TutorialOne_Sec9.ipynb)

---


In [ ]:
    def ControlLoop(self):
        self.time_ += self.dt

        t = self.time_
        d = self.stage_duration

        enable_value = 1.0

### Stage 1: Stabilization

Holds current pose using PD control.

To dive deeper into this section, open [this notebook](LeftHandRaise7DF_TutorialOne_Sec9-1.ipynb)

---

In [ ]:
        if t < d:
            self.log_stage(1, "Stabilizing (holding current pose)")

            for j in self.joints:
                q = self.low_state.motor_state[j].q
                self.low_cmd.motor_cmd[j].q = q
                self.low_cmd.motor_cmd[j].dq = 0
                self.low_cmd.motor_cmd[j].kp = self.kp
                self.low_cmd.motor_cmd[j].kd = self.kd
                self.low_cmd.motor_cmd[j].tau = 0

### Stage 2: Raise Arm

Interpolates joints to raise the arm.

To dive deeper into this section, open [this notebook](LeftHandRaise7DF_TutorialOne_Sec9-2.ipynb)

---

In [ ]:
        elif t < 2*d:
            self.log_stage(2, "Raising left arm")

            r = (t - d) / d
            for i, j in enumerate(self.joints):
                self.low_cmd.motor_cmd[j].q = self.interp(self.initial_pose[i], self.target_raise[i], r)
                self.low_cmd.motor_cmd[j].dq = 0
                self.low_cmd.motor_cmd[j].kp = self.kp
                self.low_cmd.motor_cmd[j].kd = self.kd
                self.low_cmd.motor_cmd[j].tau = 0

### Stage 3: Extend Wrist

Simulates hand opening by extending wrist joints.

To dive deeper into this section, open [this notebook](LeftHandRaise7DF_TutorialOne_Sec9-3.ipynb)

---

In [ ]:
        elif t < 3*d:
            self.log_stage(3, "Extending wrist (simulated hand open)")

            r = (t - 2*d) / d
            for i, j in enumerate(self.joints):
                self.low_cmd.motor_cmd[j].q = self.interp(self.target_raise[i], self.target_extend[i], r)
                self.low_cmd.motor_cmd[j].dq = 0
                self.low_cmd.motor_cmd[j].kp = self.kp
                self.low_cmd.motor_cmd[j].kd = self.kd
                self.low_cmd.motor_cmd[j].tau = 0

### Stage 4: Return

Returns arm to initial pose smoothly.

To dive deeper into this section, open [this notebook](LeftHandRaise7DF_TutorialOne_Sec9-4.ipynb)

---

In [ ]:
        elif t < 5*d:
            self.log_stage(4, "Returning to initial pose")

            r = (t - 3*d) / (2*d)
            for i, j in enumerate(self.joints):
                self.low_cmd.motor_cmd[j].q = self.interp(self.target_extend[i], self.initial_pose[i], r)
                self.low_cmd.motor_cmd[j].dq = 0
                self.low_cmd.motor_cmd[j].kp = self.kp
                self.low_cmd.motor_cmd[j].kd = self.kd
                self.low_cmd.motor_cmd[j].tau = 0

### Stage 5: Release Control

Gradually releases SDK control while holding final pose.

To dive deeper into this section, open [this notebook](LeftHandRaise7DF_TutorialOne_Sec9-5.ipynb)

---


In [ ]:
        elif t < 6*d:
            self.log_stage(5, "Releasing arm SDK control (holding final pose)")

            r = (t - 5*d) / d
            enable_value = (1 - r)

            for i, j in enumerate(self.joints):
                self.low_cmd.motor_cmd[j].q = self.initial_pose[i]

### Final Stage

Completes motion and disables control.

To dive deeper into this section, open [this notebook](LeftHandRaise7DF_TutorialOne_Sec9-6.ipynb)

---

In [ ]:
        else:
            if not self.done:
                print("\nMotion complete. Robot should be stable at rest.")
            self.done = True
            enable_value = 0.0

        self.low_cmd.motor_cmd[G1JointIndex.kNotUsedJoint].q = enable_value

        self.low_cmd.crc = self.crc.Crc(self.low_cmd)
        self.pub.Write(self.low_cmd)

## 10. Main Execution

Entry point: initializes system and runs controller.

To dive deeper into this section, open [this notebook](LeftHandRaise7DF_TutorialOne_Sec10.ipynb)

---

In [ ]:
if __name__ == '__main__':
    print("WARNING: Ensure robot is safely supported (gantry attached).")
    input("Press Enter to start...")

    if len(sys.argv) > 1:
        ChannelFactoryInitialize(0, sys.argv[1])
    else:
        ChannelFactoryInitialize(0)

    ctrl = LeftArmRaise()
    ctrl.Init()
    ctrl.Start()

    while True:
        time.sleep(1)
        if ctrl.done:
            sys.exit(0)

# 11. Putting It All Together

This section connects all components of the system into a single, unified understanding of how the robot is controlled.

---

## 🧠 Big Picture: The Full System

Your program is not just a script—it is a **real-time robotic control system**.

---

### Full Control Pipeline

```text
        ┌──────────────────────────────┐
        │      Your Python Code        │
        │                              │
        │  ControlLoop (50 Hz)         │
        │  - computes cmd.q            │
        │  - sets kp, kd               │
        │  - sends commands            │
        └──────────────┬───────────────┘
                       │
                       ▼
        ┌──────────────────────────────┐
        │   DDS Communication Layer    │
        │   (CycloneDDS middleware)    │
        └──────────────┬───────────────┘
                       │
                       ▼
        ┌──────────────────────────────┐
        │   G1 Onboard Controller      │
        │   - receives LowCmd          │
        │   - applies control logic    │
        └──────────────┬───────────────┘
                       │
                       ▼
        ┌──────────────────────────────┐
        │      Motor Controllers       │
        │   (QDD actuators + FOC)      │
        └──────────────┬───────────────┘
                       │
                       ▼
        ┌──────────────────────────────┐
        │        Physical Robot        │
        │   (joints, links, dynamics)  │
        └──────────────┬───────────────┘
                       │
                       ▼
        ┌──────────────────────────────┐
        │         Sensors              │
        │   (joint encoders, etc.)     │
        └──────────────┬───────────────┘
                       │
                       ▼
        ┌──────────────────────────────┐
        │        LowState (DDS)        │
        └──────────────────────────────┘
```

---

## 🔁 The Core Feedback Loop

At runtime, the system continuously executes:

```text
READ STATE → COMPUTE ACTION → SEND COMMAND → REPEAT
```

---

### In your code:

```python
state  = self.low_state
action = self.low_cmd
```

---

### This loop runs:

```text
50 times per second (every 20 ms)
```

---

## ⏱️ Timing Diagram

```text
Time →
│
├── Stage 1 (0–3s)   → Stabilize
├── Stage 2 (3–6s)   → Raise arm
├── Stage 3 (6–9s)   → Extend wrist
├── Stage 4 (9–15s)  → Return
├── Stage 5 (15–18s) → Release control
└── Final            → Stop + exit
```

---

## 🎬 Motion Composition

Your robot behavior is built from **staged trajectories**:

```text
initial_pose
   ↓
(target_raise)
   ↓
(target_extend)
   ↓
(initial_pose)
```

---

### Key concept:

> Motion is **composed of segments**, not single commands

---

## ⚙️ Control Law (What Actually Moves the Robot)

At every timestep, each joint follows:

[
\tau = K_p (q_{target} - q) + K_d (0 - \dot{q})
]

---

### Interpretation:

| Term       | Meaning               |
| ---------- | --------------------- |
| `q_target` | where you want to go  |
| `q`        | where you are         |
| `kp`       | how strongly you move |
| `kd`       | how smoothly you move |

---

### Result:

```text
Error → Torque → Motion
```

---

## 🔄 Role of Interpolation

Without interpolation:

```text
Jump → large error → large torque → unsafe motion
```

With interpolation:

```text
Small steps → smooth motion → stable behavior
```

---

### Key takeaway:

> Interpolation converts **discrete goals into continuous motion**

---

## 🧵 Concurrency Model

Your system runs multiple processes simultaneously:

```text
Thread 1: ControlLoop (actions)
Thread 2: LowStateHandler (state updates)
Main Thread: Monitoring + exit
```

---

### Important insight:

> Robotics systems are **inherently parallel**

---

## 🔌 Control Authority Lifecycle

```text
Stage 1–4 → Full SDK control
Stage 5   → Gradual release
Final     → SDK disabled
```

---

### Key concept:

> Control is not binary—it can be **blended and transferred**

---

## 🤖 Reinforcement Learning Mapping

Your system already matches the RL structure:

| RL Concept  | Your Code       |
| ----------- | --------------- |
| State       | `LowState`      |
| Action      | `LowCmd`        |
| Policy      | Control logic   |
| Environment | Robot + physics |

---

### Current (scripted policy):

```python
cmd.q = interp(...)
```

---

### RL version:

```python
cmd.q = policy(state)
```

---

### 🔥 Key Insight

> You already built an RL environment—only the policy needs to change

---

## 🧠 What Students Should Understand

This notebook teaches:

---

### 1. Control Systems

* PD control
* Feedback loops
* Stability vs responsiveness

---

### 2. Robotics Systems

* Joint-space control
* Real-time execution
* Hardware constraints

---

### 3. Software Engineering

* Multi-threading
* Message passing (DDS)
* System lifecycle

---

### 4. Motion Generation

* Interpolation
* Trajectory staging
* Behavior composition

---

### 5. RL Foundations

* state → action mapping
* control loop as environment
* policy replacement

---

## 🔥 Most Important Takeaways

### 1. Robots are controlled continuously

```text
Not: "move arm once"
But: "update commands 50 times per second"
```

---

### 2. Motion must be smooth

```text
Interpolation is essential for safety
```

---

### 3. Feedback is everything

```text
State → error → correction → motion
```

---

### 4. Control and ownership matter

```text
Who controls the robot is as important as how
```

---

### 5. This is already an RL system

```text
Only the policy is missing
```

---

## 🚀 Final Mental Model

You should now think of your system as:

```text
A real-time feedback controller
running on a distributed system
driving a physical robot
through continuous updates
```

---

## 🎓 Suggested Next Step

To extend this into a full RL module:

### 👉 Tutorial Two: “From Scripted Control to RL”

* Replace interpolation with:

  ```python
  action = policy(state)
  ```
* Define:

  * reward function
  * episode termination
* Integrate:

  * Stable-Baselines3

---

## 🏁 Final Summary

You have built:

```text
A complete, safe, real-time robot controller
with staged motion and feedback control
ready to be extended into reinforcement learning
```
